In [ ]:
from pathlib import Path

from matplotlib.dates import DateFormatter
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from scipy.stats import boxcox
from scipy.optimize import minimize
from statsforecast.models import ARIMA

from tqdm import tqdm

from constants import VALIDATION_WINDOWS

## Dynamic Regression Baseline Model

In [ ]:
class BoxCoxScaler:
    def __init__(self):
        self._lambda: float | None = None

    @property
    def is_fit(self):
        return self._lambda is not None
    
    def fit_transform(self, y: pl.Series) -> pl.Series:
        y_t, _lambda = boxcox(y.to_numpy())
        self._lambda = _lambda
        return pl.Series(name=y.name, values=y_t, dtype=pl.Float32)
    
    def transform(self, y: pl.Series) -> pl.Series:
        assert self.is_fit
        y_t = boxcox(y.to_numpy(), lmbda=self._lambda)
        return pl.Series(name=y.name, values=y_t, dtype=pl.Float32)
    
    def inverse_transform(self, y: pl.Series) -> pl.Series:
        assert self.is_fit

        if self._lambda == 0:
            y_t = np.exp(y.to_numpy())
        else:
            y_t = (y.to_numpy() * self._lambda + 1) ** (1 / self._lambda)
        return pl.Series(name=y.name, values=y_t, dtype=pl.Float32)


class CyclicFeatureTransformer:
    def __init__(self, max_harmonic: int = 5):
        self.max_harmonic = max_harmonic

    def transform(self, df: pl.DataFrame) -> pl.DataFrame:
        # Fourier cycles
        fourier_cycles = [
            ("hour", pl.col("timestamp").dt.hour(), 24),
            ("day", pl.col("timestamp").dt.day(), 7),
            ("month", pl.col("timestamp").dt.month(), 12),
        ]
        col_expr = {}
        for unit, t_expr, period in fourier_cycles:
            for k in range(1, self.max_harmonic + 1):
                angle = (2 * np.pi * k * t_expr) / period
                col_expr[f"sin_{k}_{unit}"] = np.sin(angle)
                col_expr[f"cos_{k}_{unit}"] = np.cos(angle)
        
        # Weekend indicator
        col_expr["is_weekend"] = (pl.col("timestamp").dt.weekday() >= 6).cast(pl.Float32)
        
        return df.with_columns(**col_expr)

    def get_feature_names(self) -> list[str]:
        cyclic_features = [
            f"{prefix}_{k}_{unit}"
            for unit in ["hour", "day", "month"]
            for k in range(1, self.max_harmonic + 1)
            for prefix in ["sin", "cos"]
        ]
        return cyclic_features + ["is_weekend"]


class LinearRegressionModel:
    def __init__(self, lasso: float = 0.0, include_bias: bool = True):
        self.lasso = lasso
        self.include_bias = include_bias


    def fit(self, X: np.ndarray, y: np.ndarray):
        if self.include_bias:
            X = np.hstack([np.ones((X.shape[0], 1)), X])

        # Smooth L1 approximation: sqrt(theta^2 + eps) ≈ |theta|, differentiable at 0
        _eps = 1e-8

        def f_(theta: np.ndarray):
            y_hat = np.dot(X, theta)
            l1_smooth = np.sum(np.sqrt(theta**2 + _eps))
            return np.mean((y - y_hat) ** 2) + self.lasso * l1_smooth

        x0 = np.array([0.5 for _ in range(X.shape[1])])
        result = minimize(f_, x0=x0, method="L-BFGS-B", tol=1e-6, options={"maxiter": 5000})
        if not result.success:
            raise ValueError(result.message)
        self.theta_ = result.x

        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        if self.include_bias:
            X = np.hstack([np.ones((X.shape[0], 1)), X])
        return np.dot(X, self.theta_)


class DynamicRegressionModel:
    def __init__(
        self,
        order: tuple[int, int, int],
        seasonal_order: tuple[int, int, int],
        include_bias: bool = True,
        lasso: float = 0.0,
    ):
        self.include_bias = include_bias
        self.order = order
        self.seasonal_order = seasonal_order
        self.lasso = lasso

        self._lr_model: LinearRegressionModel | None = None
        self._arima_model: ARIMA | None = None

    def fit(self, X: np.ndarray, y: np.ndarray):
        self._lr_model = LinearRegressionModel(include_bias=self.include_bias, lasso=self.lasso)
        self._lr_model.fit(X, y)

        lr_y_hat = self._lr_model.predict(X)
        y_residuals = y - lr_y_hat

        self._arima_model = ARIMA(
            order=self.order,
            season_length=24,
            seasonal_order=self.seasonal_order,
            include_mean=True,
            include_drift=False,
        )
        self._arima_model = self._arima_model.fit(y_residuals)

        return self

    def predict_in_sample(self, X: np.ndarray):
        lr_y_hat = self._lr_model.predict(X)
        arima_y_hat = self._arima_model.predict_in_sample()
        return lr_y_hat + arima_y_hat["fitted"]

    def predict(self, X: np.ndarray):
        horizon = X.shape[0]
        lr_y_hat = self._lr_model.predict(X)
        arima_y_hat = self._arima_model.predict(h=horizon)
        y_hat = lr_y_hat + arima_y_hat["mean"]

        return y_hat

## PJM Dataset

In [ ]:
PJM_SITE_NAME = "PJME"
PJM_DATA_FREQUENCY = "1h"

INPUT_PATH = Path("../../data/pjm")
OUTPUT_PATH = Path(f"../../results/pjm/sarimax/{PJM_SITE_NAME}")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

data_file_name = f"{PJM_SITE_NAME}_hourly_processed.pq"
data_file_path = INPUT_PATH / data_file_name

SITE_DF = pl.read_parquet(data_file_path)

# FE has a single value = 0
if PJM_SITE_NAME == "FE":
    SITE_DF = SITE_DF.filter(pl.col(f"{PJM_SITE_NAME}_MW").gt(0))

SITE_DF = SITE_DF.sort(by="timestamp")


### Configure

In [ ]:
MAX_FOURIER_HARMONIC = 5
TIMESTAMP_COL = "timestamp"
TARGET_COL = f"{PJM_SITE_NAME}_MW"
FORECAST_COL = f"{TARGET_COL}_FORECAST"

In [ ]:
pg_bar = tqdm(VALIDATION_WINDOWS[PJM_SITE_NAME])
for val_idx, (val_start, val_end) in enumerate(pg_bar):
    train_df = SITE_DF.filter(pl.col("timestamp").lt(val_start))
    val_df = SITE_DF.filter(pl.col("timestamp").is_between(val_start, val_end, closed="left"))

    # Feature engineering and scaling
    fft = CyclicFeatureTransformer(max_harmonic=MAX_FOURIER_HARMONIC)
    scaler = BoxCoxScaler()
    
    X_train = fft.transform(train_df)
    X_train = X_train.select(pl.col(fft.get_feature_names()))
    y_train_scaled = scaler.fit_transform(train_df[TARGET_COL])

    # Fit model
    model = DynamicRegressionModel(
        order=(1, 1, 1),
        seasonal_order=(1, 1, 1),
        include_bias=True,
        lasso=0.05,
    )
    model = model.fit(X_train.to_numpy(), y_train_scaled.to_numpy())
    
    # Get forecasts
    X_valid = fft.transform(val_df)
    X_valid = X_valid.select(pl.col(fft.get_feature_names()))
    y_hat_scaled = model.predict(X_valid.to_numpy())
    y_hat = scaler.inverse_transform(pl.Series(name=FORECAST_COL, values=y_hat_scaled))
    
    # Save forecasts
    forecast_df = val_df.with_columns(y_hat)
    forecast_output_path = f"{OUTPUT_PATH}/forecasts_{PJM_SITE_NAME}_fold_{val_idx}.pq"
    forecast_df.to_pandas().to_parquet(forecast_output_path)
    
    # Plot forecasts
    fig, ax = plt.subplots()
    
    ax.plot(forecast_df["timestamp"], forecast_df[TARGET_COL], color="black", lw=2, label="Actual")
    ax.plot(forecast_df["timestamp"], forecast_df[FORECAST_COL], color="#0072B2", lw=2, label="Forecast")
    
    ax.legend(loc=1)
    ax.grid(True, which="major", c="grey", ls="--", lw=1, alpha=0.2)
    ax.set(ylabel="Load (kWh)", title=f"Electricity Load Forecasts for Site {PJM_SITE_NAME} (Fold {val_idx})")
    
    ax.xaxis.set_major_formatter(DateFormatter("%Y-%m-%d"))
    ax.tick_params(axis='x', labelrotation=45)
    
    fig.tight_layout()
    plt.savefig(f"{OUTPUT_PATH}/forecasts_{PJM_SITE_NAME}_fold_{val_idx}.png", dpi=300);
    plt.close(fig);
